# Module 3 — Batch Pipelines + PySpark Transformations
Exam domain: **Data Processing**

Databricks notebook version — uses a `min_order_date` widget so the pipeline can
be parametrized as a Databricks Job.

In [ ]:
dbutils.widgets.text("min_order_date", "2024-01-01")
min_order_date = dbutils.widgets.get("min_order_date")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orders = spark.createDataFrame([
    (101, 1, "2024-02-01", 250.0),
    (102, 1, "2024-02-03", 80.0),
    (103, 2, "2024-02-02", 40.0),
    (104, 5, "2024-02-04", 10.0),
], ["order_id", "customer_id", "order_date", "amount"])

customers = spark.createDataFrame([
    (1, "Alice"), (2, "Bob"), (3, "Cara"),
], ["customer_id", "name"])

In [ ]:
def clean_orders(orders_df, min_date):
    return (orders_df
        .filter(F.col("amount") > 0)
        .filter(F.col("order_date") >= F.lit(min_date)))

def enrich_with_customer(orders_df, customers_df):
    return orders_df.join(customers_df, on="customer_id", how="left")

def add_running_total(df):
    w = Window.partitionBy("customer_id").orderBy("order_date")
    return df.withColumn("running_total", F.sum("amount").over(w))

clean = clean_orders(orders, min_order_date)
enriched = enrich_with_customer(clean, customers)
final_df = add_running_total(enriched)
display(final_df.orderBy("customer_id", "order_date"))

## Reacting to the widget
Change the `min_order_date` widget value at the top of the notebook (e.g. to
`2024-02-03`) and re-run — fewer rows should survive `clean_orders`.

In [ ]:
final_df.write.format("delta").mode("overwrite").saveAsTable("default.orders_pipeline_output")
print(f"Pipeline ran with min_order_date={min_order_date}, wrote", final_df.count(), "rows")

## Job-ready
This notebook can be scheduled as a Databricks Job / Workflow task, with
`min_order_date` passed in as a job parameter that overrides the widget default.